In [1]:
from google import genai
from google.genai import types
from dotenv import load_dotenv
import os

load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

for m in client.models.list():
    print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-nano-banana-2.1
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-4-argon
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/l

*Predict first. Write these three guesses on paper before you run anything:*

- ask("Say hello in exactly 5 words."). Will it obey, and will two runs give the same sentence?
   * I think so yes, since it is a simple instruction.
   * Most likely, it won't give the same sentence in two runs because, for that we need to set temparature.

- Right after that, ask("What was my previous message?"). What will it say, and why? (Think about Q5 from your notes.)
   * This Api is stateless and it won't remember the previous message unless we resend the whole text again. It won't have memory or history of conversation.

- ask("Say hello in exactly 5 words.", system="You are a pirate."). What changes?
   * I think so its behaviour will change as a pirate.

In [29]:
MODEL = "gemini-3.5-flash-lite"

import time
from google.genai import errors

MODEL = "gemini-3.5-flash-lite"
_last = 0.0
MIN_GAP = 13
_cache = {}

def _call(contents, system=None, temperature=None):
    global _last
    for attempt in range(2):
        wait = MIN_GAP - (time.time() - _last)
        if wait > 0:
            time.sleep(wait)
        _last = time.time()
        try:
            resp = client.models.generate_content(
                model=MODEL,
                contents=contents,
                config=types.GenerateContentConfig(
                    system_instruction=system,
                    temperature=temperature,
                ),
            )
            return resp.text
        except errors.APIError as e:
            if e.code != 429 or "per day" in str(e):
                raise
            time.sleep(25)
    raise RuntimeError("still rate limited")

def ask(prompt, system=None, temperature=None, use_cache=True):
    key = (MODEL, prompt, system, temperature)
    if use_cache and key in _cache:
        return _cache[key]
    text = _call(prompt, system, temperature)
    _cache[key] = text
    return text

In [25]:
print("Run 1: ", ask("Say hello in exactly 5 words"))
print("Run 2: ", ask("Say hello in exactly 5 words"))

Run 1:  Hello, how are you today?
Run 2:  Hello, how are you today?


In [26]:
print("Run 3: ", ask("What was my previous message?"))

Run 3:  This is our very first message in this conversation, so you haven't sent anything yet! What would you like to talk about?


In [27]:
print("Run 4: ", ask("Say hello in exactly 5 words.", system="You are a pirate"))

Run 4:  Ahoy there, matey! Shiver me!


In [30]:
for t in (None, 0, 2):
    print("temperature =", t)
    for _ in range(2):
        print("  ", ask("Say hello in exactly 5 words.", temperature=t))

temperature = None
   Hello there, how are you?
   Hello there, how are you?
temperature = 0
   Hello there, how are you?
   Hello there, how are you?
temperature = 2
   Hello there, how are you?
   Hello there, how are you?


In [31]:
def ask_msgs(messages, system=None, temperature=None):
    role_map = {"user": "user", "assistant": "model"}
    contents = [
        types.Content(role=role_map[m["role"]], parts=[types.Part(text=m["content"])])
        for m in messages
    ]
    return _call(contents, system, temperature)

In [38]:
SYSTEM = """You are an invoice information extraction assistant.
            Extract the invoice number, total amount, currency, and
            due date from the provided invoice text. Do not include markdown.
            total is a plain number (no commas or currency symbol).
            Convert any date format to YYYY-MM-DD. Return ONLY valid JSON with exactly the
            keys invoice_no, total, currency, due_date. If a field is missing, use null.
            currency is an ISO 4217 code such as MYR, USD or SGD (RM means MYR)."""

FEWSHOT = [
    {"role": "user", "content": "ABC Technologies Sdn Bhd\nInvoice No: INV-2026-105\nInvoice Date: 2026-10-01\nDue Date: 2026-10-31\nTotal Amount: MYR 2,500.00"},
    {"role": "assistant", "content": '{"invoice_no": "INV-2026-105", "total": 2500.0, "currency": "MYR", "due_date": "2026-10-31"}'},
]

T1 = "Delta Parts Bhd\nInvoice #: DP-88213\nDate: 15 Nov 2026\nPayment due: 14 December 2026\nAmount payable: USD 1,230.50"

print(ask_msgs(FEWSHOT + [{"role": "user", "content": T1}], SYSTEM))

{
  "invoice_no": "DP-88213",
  "total": 1230.50,
  "currency": "USD",
  "due_date": "2026-12-14"
}


In [39]:
import json
REQUIRED = {"invoice_no", "total", "currency", "due_date"}

def extract(text):
    messages = FEWSHOT + [{"role": "user", "content": text}]
    reply = ask_msgs(messages, SYSTEM)
    start = reply.find("{")
    end = reply.rfind("}")
    data = json.loads(reply[start:end+1])
    assert set(data) == REQUIRED
    return data

data = extract(T1)
print(data)
print(type(data["total"]))


{'invoice_no': 'DP-88213', 'total': 1230.5, 'currency': 'USD', 'due_date': '2026-12-14'}
<class 'float'>


*Guesses for T2 and T3*
- T2: the due date won't appear since there is no due date stated. The value written as "RM" will be return in currency.
- T3: the Total: 981 will be appear in the output.

*Now the the currency changed to MYR, it is because of the prompt not luck*

In [40]:
T2 = "Kedai Sinar\nInvoice No. 7781\nDate: 3 Oct 2026\nTotal: RM 480.00"
T3 = "Lion City Supplies\nInv: SG/2026/0042\nSubtotal: SGD 900.00\nGST 9%: SGD 81.00\nTotal: SGD 981.00\nDue: 2026-11-20"

print(extract(T2))
print(extract(T3))

{'invoice_no': '7781', 'total': 480.0, 'currency': 'MYR', 'due_date': None}
{'invoice_no': 'SG/2026/0042', 'total': 981.0, 'currency': 'SGD', 'due_date': '2026-11-20'}


In [43]:
import re

def validate(data):
    problems = []
    if not isinstance(data["total"], (int, float)):
        problems.append("total is not a number")
    if not re.fullmatch(r"[A-Z]{3}", str(data["currency"])):
        problems.append("currency is not a 3-letter ISO code")
    if data["due_date"] is not None and not re.fullmatch(r"\d{4}-\d{2}-\d{2}", data["due_date"]):
        problems.append("due_date is not YYYY-MM-DD")
    return problems

*Predict*
- good: the due date will not appear and raise the problem
- bad: the currency will output RM so raise the problem and also the due date because it is not in the format.

In [44]:
good = {"invoice_no": "X", "total": 10.0, "currency": "MYR", "due_date": None}
bad  = {"invoice_no": "X", "total": "10", "currency": "RM", "due_date": "3 Oct 2026"}
print(validate(good))
print(validate(bad))

[]
['total is not a number', 'currency is not a 3-letter ISO code', 'due_date is not YYYY-MM-DD']


In [45]:
def extract(text, retries=1):
    messages = FEWSHOT + [{"role": "user", "content": text}]
    for attempt in range(retries + 1):
        reply = ask_msgs(messages, SYSTEM)
        try:
            start = reply.find("{")
            end = reply.rfind("}")
            data = json.loads(reply[start:end+1])
            if set(data) != REQUIRED:
                problems = ["wrong keys"]
            else:
                problems = validate(data)
        except ValueError:
            problems = ["reply was not valid JSON"]
        if not problems:
            return data
        messages = messages + [
            {"role": "assistant", "content": reply},
            {"role": "user", "content": "Your reply had these problems: " + ", ".join(problems) + ". Return the corrected JSON only."},
        ]
    return None

In [46]:
replies = iter([
    '```json\n{"invoice_no": "1", "total": "5", "currency": "RM", "due_date": null}\n```',
    '{"invoice_no": "1", "total": 5.0, "currency": "MYR", "due_date": null}',
])

def fake_ask_msgs(messages, system=None, temperature=None):
    print("  call, messages so far:", len(messages))
    return next(replies)

real_ask_msgs = ask_msgs
ask_msgs = fake_ask_msgs
print(extract("anything"))
ask_msgs = real_ask_msgs

  call, messages so far: 3
  call, messages so far: 5
{'invoice_no': '1', 'total': 5.0, 'currency': 'MYR', 'due_date': None}


*Predict*

- Does the model answer the question now, or ask for something? In E14 terms, which step is this?
   * No, and in E14 terms, it is in step 2 asking the function call.
- What name and arguments do you expect inside function_calls?
   * get_exchange_rate
- Has your get_exchange_rate function run yet? Who is going to run it?
   * nothing has run yet. The model only wrote a request. Your code runs the function (E14 step 3). The model cannot run anything.

In [47]:
def get_exchange_rate(from_currency: str, to_currency: str) -> float:
    """Return how many units of to_currency one unit of from_currency buys."""
    rates = {("MYR", "SGD"): 0.30}
    return rates[(from_currency, to_currency)]

tool_config = types.GenerateContentConfig(
    tools=[get_exchange_rate],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
)

question = "How much is 350 MYR in SGD?"
time.sleep(MIN_GAP)
response = client.models.generate_content(model=MODEL, contents=question, config=tool_config)
print(response.function_calls)

[FunctionCall(
  args={
    'from_currency': 'MYR',
    'to_currency': 'SGD'
  },
  id='call_195879',
  name='get_exchange_rate'
)]


*Predict*
- What does tool result print?
   * the tool result should print 0.3
- Roughly what will the final answer say?
   * roughly the final answer say 105 MYR
- Who did the multiplication 350 × rate, your tool or the model? Why does that matter?
   * the model

In [51]:
call = response.function_calls[0]
result = get_exchange_rate(**call.args)
print("tool result:", result)

contents = [
    types.Content(role="user", parts=[types.Part(text=question)]),
    response.candidates[0].content,
    types.Content(role="user", parts=[
        types.Part.from_function_response(name=call.name, response={"result":result})
    ]),
]

time.sleep(MIN_GAP)
final = client.models.generate_content(model=MODEL, contents=contents, config=tool_config)
print(final.text)

tool result: 0.3
350 MYR is equal to 105.0 SGD (using an exchange rate of 0.3).


*Predict*

- Will the second answer know 480? Why, when the API is stateless?
   * yes i think so, since it saves it in memory but  not when the API is stateless.
- What does len(history) print?
   * 2
- Every call resends the whole history. Which two things grow as the chat gets longer?
   * history count and rate limits

In [53]:
MODEL = "gemini-3.5-flash"

history = []

def chat(user_text):
    history.append({"role":"user", "content": user_text})
    reply = ask_msgs(history, "You help with invoices and money quesions. Keep answers short")
    history.append({"role":"assistant", "content": reply})
    return reply

print(chat("My invoice total is MYR 480."))
print(chat("What total did I just tell you?"))
print(len(history))

Got it. How can I help with this invoice? 

Do you need to add tax (like SST), apply a discount, split the bill, or something else?
You told me **MYR 480**.
4


*Notes*
|Q|Answer|
|---|---|
|S21-1 Why does the invoice prompt have a system message and a few-shot example?|The system message states the rules. The example shows the exact format, and it works better than rules alone. Your run proved it.
|S21-2 Why parse and validate the reply?|Models sometimes add fences or give wrong values. T2 returned RM instead of MYR, valid JSON with a wrong value.
|S21-3 What does the retry add?|It sends the model its own bad reply plus the list of problems. If it still fails, return None and flag it for a human.
|S21-4 Who runs a tool?|Your code, never the model. The model only asks.
|S21-5 Why put the math in the tool?|Let the model decide, let code compute. The model multiplied correctly here, but you can’t depend on |that for money.
|S21-6 How does a chat remember?|It doesn’t. We keep history and resend it each time, so tokens, cost and latency grow until the context window.
|S21-7 What did the free tier teach us?|Check the real limits (5 per minute, 20 per day), pace the calls, cache while developing, and keep one _call function so the provider is easy to change.